# Neural SDE — Prepare

Оффлайн-пайплайн: тренирует модели по трём доменам (глюкоза, финансы, энергия) и складывает артефакты в `cache/`. Дашборд из этих файлов уже только читает.

Запуск целиком: выполнить все ячейки. Либо выполнить только нужный домен — каждая функция самостоятельна.

In [ ]:
# Установка зависимостей в ТЕКУЩЕЕ ядро.
# sys.executable — это python, на котором реально работает ноутбук,
# поэтому пакеты ставятся туда, куда надо, на любом интерпретаторе.
# Запусти эту ячейку первой. Идемпотентно: что уже стоит — pip пропустит.
import sys, subprocess

PKGS = [
    "torchsde", "ucimlrepo", "statsmodels",
    "scikit-learn", "scipy", "pandas", "numpy", "tqdm", "altair",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], check=True)
# yfinance обновляем принудительно: старые версии ломаются об изменившийся
# API Yahoo и отдают "possibly delisted / no price data".
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yfinance"], check=True)

print("Зависимости установлены в:", sys.executable)

In [ ]:
"""
Оффлайн-пайплайн: тренирует модели по трём доменам и складывает
артефакты (предсказания, метрики, окна) в cache/. Запускается один раз.
Дашборд из этих файлов уже только читает.

Использование:
    python prepare.py            # всё три домена
    python prepare.py --only glucose
"""
import argparse
import json
import os
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

warnings.filterwarnings("ignore")

CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)

# Реальные данные OhioT1DM лежат в data_insulin/ в корне проекта.
# Берём первый существующий путь — работает и из kt4/, и из корня.
DATA = next(
    (p for p in [Path("data_insulin"), Path("../data_insulin"), Path("data")]
     if p.exists()),
    Path("data"),
)
print("[data] папка с данными глюкозы:", DATA.resolve())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ============================================================
# 1. ГЛЮКОЗА — основная модель, Latent Neural SDE vs LSTM

In [ ]:
# ============================================================
def prepare_glucose(patient=559):
    import torchsde

    K, H = 24, 6  # 2 часа истории -> прогноз на 30 минут вперёд
    N_EPOCHS, BATCH = 60, 128
    D_LATENT = 6

    # --- данные ---
    train_xml = DATA / f"{patient}-ws-training.xml"
    test_xml = DATA / f"{patient}-ws-testing.xml"

    if not train_xml.exists():
        print(f"[glucose] {train_xml} не найден — генерирую синтетический CGM-сигнал для демо")
        df_train, df_test = _synth_cgm(n_days_train=10, n_days_test=3)
    else:
        df_train = _parse_glucose(train_xml)
        df_test = _parse_glucose(test_xml)

    print(f"[glucose] train: {len(df_train)} точек, test: {len(df_test)} точек")

    train_segs = _segments(df_train, K + H)
    test_segs_with_ts = _segments_with_ts(df_test, K + H)

    X_train_raw, Y_train_raw = _windows([s["v"] for s in train_segs], K, H)
    X_test_raw, Y_test_raw, ts_test = _windows_with_ts(test_segs_with_ts, K, H)

    g_mean = float(np.concatenate([s["v"] for s in train_segs]).mean())
    g_std = float(np.concatenate([s["v"] for s in train_segs]).std())

    X_train = (X_train_raw - g_mean) / g_std
    Y_train = (Y_train_raw - g_mean) / g_std
    X_test = (X_test_raw - g_mean) / g_std

    X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1).to(DEVICE)
    Y_train_t = torch.tensor(Y_train, dtype=torch.float32).to(DEVICE)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1).to(DEVICE)

    # --- LSTM-бейзлайн ---
    print("[glucose] обучаю LSTM...")
    lstm = _LSTM(H).to(DEVICE)
    _train_lstm(lstm, X_train_t, Y_train_t, epochs=N_EPOCHS, bs=BATCH)

    # --- Neural SDE ---
    print("[glucose] обучаю Neural SDE...")
    nsde = _NSDE(D_LATENT, K, H).to(DEVICE)
    ts_future = torch.tensor([0.0] + [float(i + 1) for i in range(H)], device=DEVICE)
    _train_nsde(nsde, X_train_t, Y_train_t, ts_future, epochs=N_EPOCHS, bs=BATCH)

    # --- предсказания на тесте ---
    print("[glucose] инференс на тесте...")
    lstm.eval(); nsde.eval()
    with torch.no_grad():
        lstm_mu, lstm_ls = lstm(X_test_t)
        lstm_mu = lstm_mu.cpu().numpy() * g_std + g_mean
        lstm_sigma = (torch.exp(lstm_ls) * g_std).cpu().numpy()

        N_MC = 200
        nsde_samples = np.zeros((len(X_test_t), H, N_MC), dtype=np.float32)
        for m in tqdm(range(N_MC), desc="MC"):
            mu_y, ls_y, _, _ = nsde(X_test_t, ts_future, mc_z0=True)
            eps = torch.randn_like(mu_y)
            y_sample = mu_y + torch.exp(ls_y) * eps
            nsde_samples[:, :, m] = y_sample.cpu().numpy() * g_std + g_mean

    # --- метрики на горизонте +30 минут ---
    metrics = _glucose_metrics(Y_test_raw, lstm_mu, lstm_sigma, nsde_samples)

    np.savez_compressed(
        CACHE / "glucose.npz",
        X_test_raw=X_test_raw.astype(np.float32),
        Y_test_raw=Y_test_raw.astype(np.float32),
        ts_test=ts_test.astype("datetime64[s]"),
        lstm_mu=lstm_mu.astype(np.float32),
        lstm_sigma=lstm_sigma.astype(np.float32),
        nsde_samples=nsde_samples,
        g_mean=g_mean, g_std=g_std,
        patient=patient,
    )
    with open(CACHE / "glucose_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"[glucose] готово — {len(X_test_raw)} тестовых окон")


def _parse_glucose(xml_path):
    root = ET.parse(xml_path).getroot()
    rows = []
    for ev in root.find("glucose_level").findall("event"):
        rows.append({
            "ts": pd.to_datetime(ev.get("ts"), format="%d-%m-%Y %H:%M:%S", errors="coerce"),
            "value": float(ev.get("value")),
        })
    return pd.DataFrame(rows).dropna().sort_values("ts").reset_index(drop=True)


def _synth_cgm(n_days_train, n_days_test):
    # Похожий на CGM сигнал: базовый уровень + суточный цикл + приём пищи + шум.
    # Нужен только для демо, если у пользователя нет OhioT1DM.
    def gen(n_days, start):
        n = n_days * 24 * 12  # 5-минутные шаги
        t = np.arange(n)
        hours = (t * 5 / 60) % 24
        base = 110 + 25 * np.sin(2 * np.pi * (hours - 6) / 24)
        meals = np.zeros(n)
        for day in range(n_days):
            for meal_hour in [8, 13, 19]:
                idx = day * 288 + meal_hour * 12 + np.random.randint(-6, 6)
                if idx < n:
                    spike = 60 * np.exp(-((t - idx) / 8) ** 2 * 0.3) * (t >= idx)
                    meals += spike
        signal = base + meals + np.random.normal(0, 5, n)
        signal = np.clip(signal, 50, 350)
        ts = pd.date_range(start, periods=n, freq="5min")
        return pd.DataFrame({"ts": ts, "value": signal})
    return gen(n_days_train, "2024-01-01"), gen(n_days_test, "2024-02-01")


def _segments(df, min_len, tol_min=6):
    ts = df["ts"].values
    vals = df["value"].values
    dt_min = np.diff(ts).astype("timedelta64[s]").astype(float) / 60.0
    breaks = np.where(dt_min > tol_min)[0]
    bnd = np.concatenate([[0], breaks + 1, [len(vals)]])
    out = []
    for i in range(len(bnd) - 1):
        v = vals[bnd[i]:bnd[i + 1]]
        if len(v) >= min_len:
            out.append({"v": v, "ts": ts[bnd[i]:bnd[i + 1]]})
    return out


def _segments_with_ts(df, min_len, tol_min=6):
    return _segments(df, min_len, tol_min)


def _windows(segs, K, H):
    X, Y = [], []
    for s in segs:
        for i in range(len(s) - K - H + 1):
            X.append(s[i:i + K])
            Y.append(s[i + K:i + K + H])
    return np.array(X), np.array(Y)


def _windows_with_ts(segs, K, H):
    X, Y, TS = [], [], []
    for s in segs:
        v, t = s["v"], s["ts"]
        for i in range(len(v) - K - H + 1):
            X.append(v[i:i + K])
            Y.append(v[i + K:i + K + H])
            TS.append(t[i + K])  # время начала горизонта прогноза
    return np.array(X), np.array(Y), np.array(TS)


class _LSTM(nn.Module):
    def __init__(self, H, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.head_mu = nn.Linear(hidden, H)
        self.head_ls = nn.Linear(hidden, H)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = h[-1]
        return self.head_mu(h), self.head_ls(h).clamp(-3.0, 1.5)


def _train_lstm(model, X, Y, epochs, bs, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.6)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(X))
        for b in range(max(1, len(X) // bs)):
            ix = perm[b * bs:(b + 1) * bs]
            mu, ls = model(X[ix])
            var = torch.exp(2 * ls)
            nll = 0.5 * (np.log(2 * np.pi) + 2 * ls + (Y[ix] - mu) ** 2 / var)
            opt.zero_grad(); nll.mean().backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        sched.step()


class _LatentSDE(nn.Module):
    sde_type = "ito"
    noise_type = "diagonal"

    def __init__(self, D, hidden=48):
        super().__init__()
        self.f_net = nn.Sequential(nn.Linear(D, hidden), nn.Tanh(),
                                   nn.Linear(hidden, hidden), nn.Tanh(),
                                   nn.Linear(hidden, D))
        self.g_net = nn.Sequential(nn.Linear(D, hidden), nn.Tanh(),
                                   nn.Linear(hidden, D))

    def f(self, t, y): return self.f_net(y)
    def g(self, t, y): return torch.exp(self.g_net(y).clamp(-3, 1)) + 1e-3


class _NSDE(nn.Module):
    def __init__(self, D, K, H, hidden=48):
        super().__init__()
        self.D, self.K, self.H = D, K, H
        self.encoder = nn.GRU(1, hidden, batch_first=True)
        self.drop = nn.Dropout(0.2)
        self.head_mu = nn.Linear(hidden, D)
        self.head_lv = nn.Linear(hidden, D)
        self.sde = _LatentSDE(D, hidden)
        self.decoder = nn.Sequential(nn.Linear(D, hidden), nn.Tanh(),
                                     nn.Linear(hidden, 2))

    def encode(self, x):
        _, h = self.encoder(x)
        h = self.drop(h.squeeze(0))
        return self.head_mu(h), self.head_lv(h).clamp(-4.0, 2.0)

    def forward(self, x, ts, mc_z0=True):
        import torchsde
        mu0, lv0 = self.encode(x)
        std0 = torch.exp(0.5 * lv0)
        z0 = mu0 + std0 * torch.randn_like(mu0) if mc_z0 else mu0
        zs = torchsde.sdeint(self.sde, z0, ts, method="euler", dt=0.1)
        decoded = self.decoder(zs[1:])
        return (decoded[..., 0].transpose(0, 1),
                decoded[..., 1].clamp(-3, 1.5).transpose(0, 1),
                mu0, lv0)


def _train_nsde(model, X, Y, ts, epochs, bs, lr=2e-3, beta_kl=1.0):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.6)
    for ep in range(epochs):
        warm = min(1.0, 0.2 + 0.8 * ep / 10.0)
        model.train()
        perm = torch.randperm(len(X))
        for b in range(max(1, len(X) // bs)):
            ix = perm[b * bs:(b + 1) * bs]
            mu_y, ls_y, mu0, lv0 = model(X[ix], ts)
            var = torch.exp(2 * ls_y)
            nll = 0.5 * (np.log(2 * np.pi) + 2 * ls_y + (Y[ix] - mu_y) ** 2 / var)
            kl = 0.5 * (mu0.pow(2) + torch.exp(lv0) - lv0 - 1).sum(dim=-1)
            loss = nll.mean() + beta_kl * warm * kl.mean() / max(Y[ix].shape[1], 1)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        sched.step()


def _glucose_metrics(Y_test_raw, lstm_mu, lstm_sigma, nsde_samples):
    # Считаем для горизонта t+30 мин (последний шаг) — стандартный CGM-бенчмарк
    H_last = Y_test_raw.shape[1] - 1
    y = Y_test_raw[:, H_last]
    lm, ls = lstm_mu[:, H_last], lstm_sigma[:, H_last]
    ns = nsde_samples[:, H_last, :]

    def crps_g(y, m, s):
        z = (y - m) / s
        return float(np.mean(s * (z * (2 * stats.norm.cdf(z) - 1)
                                  + 2 * stats.norm.pdf(z) - 1 / np.sqrt(np.pi))))

    def crps_e(y, S):
        n = S.shape[-1]
        Ss = np.sort(S, axis=-1)
        t1 = np.mean(np.abs(S - y[..., None]), axis=-1)
        w = (2 * np.arange(n) - n + 1) / (n * (n - 1))
        return float(np.mean(t1 - (Ss * w).sum(axis=-1)))

    def cov_g(m, s, y, a):
        z = stats.norm.ppf(1 - (1 - a) / 2)
        return float(np.mean((y >= m - z * s) & (y <= m + z * s)))

    def cov_s(S, y, a):
        lo = np.quantile(S, (1 - a) / 2, axis=-1)
        hi = np.quantile(S, 1 - (1 - a) / 2, axis=-1)
        return float(np.mean((y >= lo) & (y <= hi)))

    nsde_mean = ns.mean(axis=-1)
    return {
        "horizon_min": 30,
        "n_test": int(len(y)),
        "lstm": {
            "mae": float(np.mean(np.abs(y - lm))),
            "rmse": float(np.sqrt(np.mean((y - lm) ** 2))),
            "crps": crps_g(y, lm, ls),
            "cov80": cov_g(lm, ls, y, 0.80),
            "cov90": cov_g(lm, ls, y, 0.90),
            "cov95": cov_g(lm, ls, y, 0.95),
        },
        "nsde": {
            "mae": float(np.mean(np.abs(y - nsde_mean))),
            "rmse": float(np.sqrt(np.mean((y - nsde_mean) ** 2))),
            "crps": crps_e(y, ns),
            "cov80": cov_s(ns, y, 0.80),
            "cov90": cov_s(ns, y, 0.90),
            "cov95": cov_s(ns, y, 0.95),
        },
    }

In [ ]:
# ============================================================
# 2. ФИНАНСЫ — AAPL, для страницы сравнения

In [ ]:
# ============================================================
def prepare_finance():
    import time
    import yfinance as yf
    from statsmodels.tsa.statespace.sarimax import SARIMAX

    WINDOW = 20
    N_STEPS, DT = 64, 1.0 / 64
    SQRT_DT = DT ** 0.5
    N_ITER, BATCH, LR = 3000, 128, 1e-3
    OBS_STD = 0.1

    def _dl(ticker, tries=4):
        # Yahoo иногда отвечает пустым телом (троттлинг) — повторяем с паузой.
        # Свежий yfinance отдаёт MultiIndex-колонки даже для одного тикера —
        # выпрямляем до простых имён (Close, Volume, ...).
        for k in range(tries):
            d = yf.download(ticker, start="2018-01-01", end="2024-12-31",
                            auto_adjust=True, progress=False)
            if d is not None and not d.empty:
                if isinstance(d.columns, pd.MultiIndex):
                    d.columns = d.columns.get_level_values(0)
                return d
            print(f"[finance] {ticker}: пусто, попытка {k + 1}/{tries}, жду...")
            time.sleep(3 * (k + 1))
        raise RuntimeError(
            f"[finance] Yahoo не отдал данные по {ticker} за {tries} попыток. "
            f"Чаще всего это троттлинг — подожди минуту и запусти ячейку заново."
        )

    print("[finance] загружаю данные с Yahoo...")
    stock = _dl("AAPL")
    qqq = _dl("QQQ")
    vix = _dl("^VIX")

    df = pd.DataFrame(index=stock.index)
    df["close"] = stock["Close"]
    df["volume"] = stock["Volume"]
    df["qqq"] = qqq["Close"]
    df["vix"] = vix["Close"]
    df = df.dropna()

    df["r"] = np.log(df["close"] / df["close"].shift(1))
    df["qqq_r"] = np.log(df["qqq"] / df["qqq"].shift(1))
    df["vix_r"] = np.log(df["vix"] / df["vix"].shift(1))
    df["vol_r"] = np.log(df["volume"] / df["volume"].shift(1))
    df["roll_std"] = df["r"].rolling(WINDOW).std()
    df["roll_mean"] = df["r"].rolling(WINDOW).mean()
    df = df.dropna()

    r = df["r"].values
    X, Y = [], []
    for i in range(len(df) - WINDOW - 1):
        ctx = np.concatenate([
            r[i:i + WINDOW],
            [df["roll_std"].iloc[i + WINDOW - 1],
             df["roll_mean"].iloc[i + WINDOW - 1],
             df["qqq_r"].iloc[i + WINDOW - 1],
             df["vix_r"].iloc[i + WINDOW - 1],
             df["vol_r"].iloc[i + WINDOW - 1]],
        ])
        X.append(ctx); Y.append(r[i + WINDOW])
    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.float32)

    n_tr = int(len(X) * 0.8)
    X_tr, X_te = X[:n_tr], X[n_tr:]
    Y_tr, Y_te = Y[:n_tr], Y[n_tr:]
    dates_te = df.index[WINDOW + n_tr:WINDOW + n_tr + len(X_te)]

    xs, ys = StandardScaler(), StandardScaler()
    X_tr_s = xs.fit_transform(X_tr); X_te_s = xs.transform(X_te)
    Y_tr_s = ys.fit_transform(Y_tr.reshape(-1, 1)).flatten()

    print("[finance] ARIMA + SARIMA...")
    arima = SARIMAX(Y_tr, order=(2, 0, 2), enforce_stationarity=False,
                    enforce_invertibility=False).fit(disp=False)
    arima_pred = np.asarray(arima.forecast(len(Y_te)))
    sarima = SARIMAX(Y_tr, order=(2, 0, 2), seasonal_order=(1, 0, 1, 5),
                     enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    sarima_pred = np.asarray(sarima.forecast(len(Y_te)))

    print("[finance] Neural SDE...")
    INPUT = X_tr_s.shape[1]

    class FDrift(nn.Module):
        def __init__(self, in_dim, h=64):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(in_dim + 1, h), nn.Tanh(),
                                     nn.Linear(h, h), nn.Tanh(),
                                     nn.Linear(h, 1))

        def forward(self, x, c):
            return self.net(torch.cat([x, c], dim=-1))

    net = FDrift(INPUT).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=LR)

    for it in tqdm(range(N_ITER), desc="NSDE"):
        idx = np.random.randint(0, len(X_tr_s), BATCH)
        xb = torch.tensor(X_tr_s[idx], device=DEVICE)
        yb = torch.tensor(Y_tr_s[idx, None], device=DEVICE)
        X_ = torch.zeros(BATCH, 1, device=DEVICE)
        kl = torch.zeros(BATCH, device=DEVICE)
        for _ in range(N_STEPS):
            drift = net(X_, xb)
            kl = kl + 0.5 * (drift ** 2).sum(-1) * DT
            X_ = X_ + drift * DT + SQRT_DT * torch.randn_like(X_)
        ll = -0.5 / OBS_STD ** 2 * ((X_ - yb) ** 2).sum(-1)
        loss = -(ll - 0.01 * kl).mean()
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
        opt.step()

    with torch.no_grad():
        xb = torch.tensor(X_te_s, device=DEVICE)
        X_ = torch.zeros(len(X_te_s), 1, device=DEVICE)
        for _ in range(N_STEPS):
            drift = net(X_, xb)
            X_ = X_ + drift * DT + SQRT_DT * torch.randn_like(X_)
        pred_s = X_[:, 0].cpu().numpy()
    nsde_pred = ys.inverse_transform(pred_s.reshape(-1, 1)).flatten()

    def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))

    np.savez_compressed(
        CACHE / "finance.npz",
        Y_test=Y_te.astype(np.float32),
        nsde_pred=nsde_pred.astype(np.float32),
        arima_pred=arima_pred.astype(np.float32),
        sarima_pred=sarima_pred.astype(np.float32),
        dates=np.array(dates_te.astype(str)),
    )
    with open(CACHE / "finance_metrics.json", "w") as f:
        json.dump({
            "nsde_rmse": rmse(Y_te, nsde_pred),
            "arima_rmse": rmse(Y_te, arima_pred),
            "sarima_rmse": rmse(Y_te, sarima_pred),
            "n_test": int(len(Y_te)),
        }, f, indent=2)
    print("[finance] готово")

In [ ]:
# ============================================================
# 3. ЭНЕРГОПОТРЕБЛЕНИЕ — UCI Household

In [ ]:
def prepare_energy():
    from statsmodels.tsa.statespace.sarimax import SARIMAX

    # Локальный файл UCI Household Power Consumption (лежит в electric-data/).
    # Берём первый существующий путь — работает и из kt4/, и из корня.
    elec_dir = next(
        (p for p in [Path("electric-data"), Path("../electric-data")] if p.exists()),
        None,
    )
    if elec_dir is None:
        raise FileNotFoundError(
            "Папка electric-data не найдена (ни в kt4/, ни в корне проекта)."
        )
    txts = sorted(elec_dir.glob("household_power_consumption*.txt"))
    if not txts:
        raise FileNotFoundError(
            f"В {elec_dir.resolve()} нет файла household_power_consumption*.txt"
        )
    print(f"[energy] читаю локальный файл: {txts[0].resolve()}")

    # Формат UCI: ';'-разделитель, пропуски обозначены '?'.
    raw = pd.read_csv(txts[0], sep=";", na_values="?", low_memory=False)
    raw["dt"] = pd.to_datetime(raw["Date"] + " " + raw["Time"], format="%d/%m/%Y %H:%M:%S")
    raw["Global_active_power"] = pd.to_numeric(raw["Global_active_power"], errors="coerce")
    raw = raw.dropna(subset=["Global_active_power"]).sort_values("dt").reset_index(drop=True)

    daily = raw.set_index("dt")["Global_active_power"].resample("D").sum()
    daily = daily[daily > 10]
    series = daily.values.astype(np.float32)
    print(f"[energy] дней после ресемплинга: {len(series)}")

    WINDOW = 14
    log_s = np.log1p(series)
    X, Y = [], []
    for i in range(len(log_s) - WINDOW - 1):
        past = log_s[i:i + WINDOW]
        day_idx = i + WINDOW
        dow, month = day_idx % 7, (day_idx // 30) % 12
        extra = np.array([
            np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7),
            np.sin(2 * np.pi * month / 12), np.cos(2 * np.pi * month / 12),
            past.mean(), past.std() + 1e-6,
        ], dtype=np.float32)
        X.append(np.concatenate([past, extra]).astype(np.float32))
        Y.append(log_s[i + WINDOW])
    X = np.array(X); Y = np.array(Y)
    n_tr = int(len(X) * 0.8)
    X_tr, X_te = X[:n_tr], X[n_tr:]
    Y_tr, Y_te = Y[:n_tr], Y[n_tr:]

    xs, ys = StandardScaler(), StandardScaler()
    X_tr_s = xs.fit_transform(X_tr); X_te_s = xs.transform(X_te)
    Y_tr_s = ys.fit_transform(Y_tr.reshape(-1, 1)).flatten()

    print("[energy] SARIMA...")
    sar = SARIMAX(Y_tr, order=(2, 0, 2), seasonal_order=(1, 0, 1, 5),
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    sar_pred = np.asarray(sar.forecast(len(Y_te)))

    print("[energy] Neural SDE...")
    N_STEPS, DT = 64, 1.0 / 64
    SQRT_DT = DT ** 0.5
    OBS_STD = 0.05
    INPUT = X_tr_s.shape[1]

    class EDrift(nn.Module):
        def __init__(self, c, h=128):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(1 + c, h), nn.SiLU(),
                                     nn.Linear(h, h), nn.SiLU(),
                                     nn.Linear(h, h // 2), nn.SiLU(),
                                     nn.Linear(h // 2, 1))

        def forward(self, x, c):
            return self.net(torch.cat([x, c], dim=-1))

    net = EDrift(INPUT).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=3e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=4000, eta_min=1e-5)

    for it in tqdm(range(4000), desc="NSDE"):
        beta = min(0.05, max(0.0, (it - 500) / 1000 * 0.05))
        idx = np.random.randint(0, len(X_tr_s), 128)
        cb = torch.tensor(X_tr_s[idx], device=DEVICE)
        yb = torch.tensor(Y_tr_s[idx], device=DEVICE)
        X_ = torch.zeros(128, 1, device=DEVICE)
        kl = torch.zeros(128, device=DEVICE)
        for _ in range(N_STEPS):
            drift = net(X_, cb)
            kl = kl + 0.5 * (drift ** 2).squeeze(-1) * DT
            X_ = X_ + drift * DT + SQRT_DT * torch.randn_like(X_)
        ll = -0.5 / OBS_STD ** 2 * ((X_.squeeze(-1) - yb) ** 2)
        loss = -(ll - beta * kl).mean()
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
        opt.step(); sched.step()

    with torch.no_grad():
        cb = torch.tensor(X_te_s, device=DEVICE)
        preds = []
        for _ in range(64):
            X_ = torch.zeros(len(X_te_s), 1, device=DEVICE)
            for _ in range(N_STEPS):
                drift = net(X_, cb)
                X_ = X_ + drift * DT + SQRT_DT * torch.randn_like(X_)
            preds.append(X_.squeeze(-1).cpu().numpy())
        pred_mean = np.stack(preds).mean(0)
        pred_std = np.stack(preds).std(0)

    pred_log = ys.inverse_transform(pred_mean.reshape(-1, 1)).flatten()
    lo_log = ys.inverse_transform((pred_mean - pred_std).reshape(-1, 1)).flatten()
    hi_log = ys.inverse_transform((pred_mean + pred_std).reshape(-1, 1)).flatten()
    nsde = np.expm1(pred_log)
    Y_te_orig = np.expm1(Y_te)
    sar_orig = np.expm1(sar_pred)

    def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))

    np.savez_compressed(
        CACHE / "energy.npz",
        Y_test=Y_te_orig.astype(np.float32),
        nsde_pred=nsde.astype(np.float32),
        sarima_pred=sar_orig.astype(np.float32),
        ci_lo=np.expm1(lo_log).astype(np.float32),
        ci_hi=np.expm1(hi_log).astype(np.float32),
    )
    with open(CACHE / "energy_metrics.json", "w") as f:
        json.dump({
            "nsde_rmse": rmse(Y_te_orig, nsde),
            "sarima_rmse": rmse(Y_te_orig, sar_orig),
            "n_test": int(len(Y_te_orig)),
        }, f, indent=2)
    print("[energy] готово")

In [ ]:
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--only", choices=["glucose", "finance", "energy"], default=None)
    # parse_known_args, а не parse_args: в Jupyter в argv лежат служебные
    # аргументы ядра (-f .../kernel.json), которые иначе ломают разбор.
    args, _ = ap.parse_known_args()

    if args.only in (None, "glucose"):
        prepare_glucose()
    if args.only in (None, "finance"):
        prepare_finance()
    if args.only in (None, "energy"):
        prepare_energy()
    print("\nГотово. Можно запускать: streamlit run app.py")